# Fine-Tuning PhoBERT Multi-Label Moderation (v1.0)

Notebook này nạp tập dữ liệu kiểm duyệt đã phân chia theo tỷ lệ học sâu chuẩn **70% Train (25,125 mẫu)**, **15% Val (5,384 mẫu)**, và **15% Test (5,384 mẫu)** trực tiếp từ thư mục làm việc hiện tại (Root working directory / Google Colab).

### 🏷️ Taxonomy 4 Nhãn AI Ngữ Cảnh:
- `0: CLEAN` - Nội dung an toàn, tích cực.
- `1: PROFANITY_VENTING` - Từ chửi thề nhẹ / Bộc phát xả stress.
- `2: HATE_SPEECH` - Ngôn từ thù ghét / Công kích cá nhân.
- `3: SELF_HARM_CRISIS` - Ý định tự hại / Trầm cảm tuyệt vọng.
*(Lưu ý: Nhãn `4: ILLEGAL_PORN` được đảm bảo bởi Lớp 1 Hard-Block Engine bằng Regex/Rules)*

In [ ]:
# Step 1: Kiểm tra môi trường & Cài đặt các thư viện cần thiết
!pip install -q transformers datasets torch accelerate scikit-learn matplotlib seaborn

In [ ]:
# Step 2: Nạp các tập train.json, val.json, test.json trực tiếp từ Root directory (Google Colab / Current Dir)
import os
import json
from pathlib import Path

def find_data_file(filename):
    candidates = [
        Path(filename),                        # Direct root/current dir
        Path("../data") / filename,            # Colab parent data
        Path("../../data") / filename,         # Relative pipeline data
        Path("moderation/data") / filename     # Relative root data
    ]
    for p in candidates:
        if p.exists():
            return p
    return Path(filename)

train_path = find_data_file("train.json")
val_path = find_data_file("val.json")
test_path = find_data_file("test.json")

with open(train_path, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(val_path, "r", encoding="utf-8") as f:
    val_data = json.load(f)
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Loaded Train samples from {train_path}: {len(train_data):,}")
print(f"Loaded Val samples   from {val_path}  : {len(val_data):,}")
print(f"Loaded Test samples  from {test_path} : {len(test_data):,}")

In [ ]:
# Step 3: Tokenization & Chuyển đổi sang định dạng HuggingFace Dataset
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

MODEL_NAME = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_ds = Dataset.from_dict({"text": [x["text"] for x in train_data], "label": [x["label"] for x in train_data]}).map(tokenize_fn, batched=True)
val_ds = Dataset.from_dict({"text": [x["text"] for x in val_data], "label": [x["label"] for x in val_data]}).map(tokenize_fn, batched=True)
test_ds = Dataset.from_dict({"text": [x["text"] for x in test_data], "label": [x["label"] for x in test_data]}).map(tokenize_fn, batched=True)

# Khởi tạo mô hình PhoBERT-base-v2 cho 4 nhãn kiểm duyệt ngữ cảnh
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)

In [ ]:
# Step 4: Cấu hình tham số huấn luyện (TrainingArguments) & Early Stopping
eval_strat_key = "eval_strategy" if hasattr(TrainingArguments, "eval_strategy") else "evaluation_strategy"

args_dict = {
    "output_dir": "./results_phobert_moderation_v1.0",
    "num_train_epochs": 5,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-5,
    "warmup_steps": 500,
    "weight_decay": 0.01,
    eval_strat_key: "epoch",
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "fp16": torch.cuda.is_available(),
    "logging_steps": 100
}

training_args = TrainingArguments(**args_dict)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": macro_f1, "weighted_f1": weighted_f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# Step 5: Bắt đầu Huấn luyện Mô hình PhoBERT Moderation
print("Starting PhoBERT Multi-Label Moderation Fine-Tuning...")
trainer.train()

In [ ]:
# Step 6: Đánh giá chi tiết mô hình trên tập Test độc lập (5,384 mẫu)
test_predictions = trainer.predict(test_ds)
preds = np.argmax(test_predictions.predictions, axis=1)
labels = test_predictions.label_ids

target_names = ["0: CLEAN", "1: PROFANITY_VENTING", "2: HATE_SPEECH", "3: SELF_HARM_CRISIS"]
print("\n=================== TEST SET CLASSIFICATION REPORT ===================\n")
print(classification_report(labels, preds, target_names=target_names, digits=4))

In [ ]:
# Step 7: Vẽ Confusion Matrix chuyên nghiệp cho báo cáo khoa học / hội đồng
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix - PhoBERT Multi-Label Moderation v1.0")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Step 8: Lưu mô hình & tokenizer đã fine-tune vào thư mục local
save_directory = "./saved_phobert_moderation_v1.0"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model and tokenizer successfully saved to {save_directory}")

In [ ]:
# Step 9: Đóng gói ZIP và Tải trực tiếp về máy tính
import shutil
from google.colab import files

zip_name = "saved_phobert_moderation_v1.0"
shutil.make_archive(zip_name, 'zip', save_directory)
print(f"Created {zip_name}.zip successfully!")

# Tải file zip về máy tính
files.download(f"{zip_name}.zip")